In [ ]:
# Mit komplett numpy logik
import numpy as np

# Fläche in 100 m², Zimmer
X = np.array([[21.0, 3],
              [14.2, 3],
              [15.3, 2],
              [ 8.5, 1],
              [ 3.0, 1]])

# Preis in Tsd.
y = np.array([460.0, 232.0, 315.0, 178.0, 90.0])
x = X[:, 0]
neu = np.array([20, 30, 40])
grad = 3
iterations = 3000
w_in = np.zeros(grad)
b_in = 0
alpha = 0.1

def erweitere_merkmale(x, grad):
    X = np.column_stack([x**i for i in range(1, grad + 1)])
    return X

def skaliere(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    X_skal = (X - mu) / sigma

    return X_skal, mu, sigma

#X_skal = skaliere(X)[0]

def cost_gradient(X, y, w, b, alpha, iterations):
    m = len(X)
    cost_pattern = []
    for iter in range(iterations):
        vorhersage = X @ w + b              # alle m Zeilen auf einmal → (m,)
        err = vorhersage - y                # Fehler je Zeile → (m,)
        cost = (err ** 2).sum() / (2 * m)   # eine Zahl
        dw = X.T @ err / m                  # je Spalte: Summe(err * x_j) / m → (n,)
        db = err.mean()                     # Summe(err) / m
        cost_pattern.append(cost)
        w = w - alpha * dw
        b = b - alpha * db

    return w, b, cost_pattern

def vorhersage(neu, grad, mu, sigma, w, b):
    P_neu = erweitere_merkmale(neu, grad)
    P_neu_skal = (P_neu - mu) / sigma
    preis = P_neu_skal @ w + b
    return preis

P = erweitere_merkmale(x, grad)
P_skal, mu, sigma = skaliere(P)
w, b, cost_pattern = cost_gradient(P_skal, y, w_in, b_in, alpha, iterations)
preis_prognose = vorhersage(neu, grad, mu, sigma, w, b)

print(w)
print(b)
print(P_skal)
print(preis_prognose)
for i in range(0, len(cost_pattern), 400):
    print(f"Iteration {i:5}: Cost = {cost_pattern[i]:10.2f}")

[58.86702811 11.7074309  56.15259273]
254.9999999999999
[[ 1.39812449  1.66868358  1.82784159]
 [ 0.29263071  0.06720124 -0.12389086]
 [ 0.47146058  0.28431396  0.09523559]
 [-0.6340332  -0.79850648 -0.81003685]
 [-1.52818258 -1.2216923  -0.98914946]]
[ 425.0944186   885.43748271 1669.79240712]
Iteration     0: Cost =   40443.30
Iteration   400: Cost =     201.29
Iteration   800: Cost =     200.60
Iteration  1200: Cost =     199.93
Iteration  1600: Cost =     199.30
Iteration  2000: Cost =     198.68
Iteration  2400: Cost =     198.09
Iteration  2800: Cost =     197.52


In [2]:
from sklearn.linear_model import LinearRegression

modell = LinearRegression()
modell.fit(P_skal, y)                     # dieselben skalierten Merkmale wie dein GD

print("GD      w:", w.round(2), " b:", round(b, 2))
print("sklearn w:", modell.coef_.round(2), " b:", round(modell.intercept_, 2))

neu_skal = (erweitere_merkmale(neu, grad) - mu) / sigma   # eingefrorenes mu/sigma
print("GD      Vorhersage:", vorhersage(neu, grad, mu, sigma, w, b).round(1))
print("sklearn Vorhersage:", modell.predict(neu_skal).round(1))
print(np.corrcoef(P_skal.T).round(3))

GD      w: [58.87 11.71 56.15]  b: 255.0
sklearn w: [ 145.1  -188.02  173.75]  b: 255.0
GD      Vorhersage: [ 425.1  885.4 1669.8]
sklearn Vorhersage: [ 422.9 1036.9 2353.4]
[[1.    0.972 0.918]
 [0.972 1.    0.985]
 [0.918 0.985 1.   ]]
